In [6]:
import requests
import json
import pandas as pd
import pymssql
from sqlalchemy import create_engine

In [7]:
with open("../config/api_config.json", 'r') as f:
    api_config = json.load(f)

api_key = api_config["api_key_google_maps"]
#print(api_config)


### Retrieve the waypoints from azure cloud

In [8]:
# Load configuration from config/db_config.json
with open('../config/db_config.json', 'r') as f:
    db_config = json.load(f)

# Get database credentials
server = db_config['server']
database = db_config['database']
db_user = db_config['db_user']
db_password = db_config['db_password']# Connect to SQL Database
conn = pymssql.connect(server, db_user, db_password, database)

# Create connection string for SQLAlchemy
connection_string = f"mssql+pymssql://{db_user}:{db_password}@{server}/{database}"
engine = create_engine(connection_string)

In [10]:
# SQL-Abfrage für die gewünschten Spalten
query = "SELECT ID, LAT, LON FROM OVRP_waypoints"

# Daten aus der Azure SQL-Datenbank laden
waypoints = pd.read_sql_query(query, con=engine)

# IDs direkt in eine Liste umwandeln
# list_hikingroute_ids = df_hikingroute_ids['id'].tolist()

# Ergebnis anzeigen
# print(list_hikingroute_ids)
print(waypoints.head)

<bound method NDFrame.head of               ID        LAT       LON
0       28022787  46.871293  6.581740
1       28022788  46.874111  6.589236
2       28022789  46.878365  6.598630
3       33636662  46.867863  6.573559
4       36789486  46.880193  6.612009
...          ...        ...       ...
31760  497874972  47.713981  8.889910
31761  497874993  47.711298  8.889283
31762  497874994  47.713006  8.889728
31763  497874995  47.710600  8.888011
31764  497874996  47.709732  8.886718

[31765 rows x 3 columns]>


In [4]:
# Beispiel-DataFrame mit IDs, Latitude und Longitude
waypoints = pd.DataFrame({
    "id": [1, 2, 3],
    "latitude": [46.8609414, 46.9481, 46.2044],
    "longitude": [6.6144089, 7.4474, 6.1432]
})


In [13]:
# Empty list to store the results
results = []

# Iterate over the waypoints DataFrame
for index, row in waypoints.iterrows():
    latitude = row["LAT"]
    longitude = row["LON"]
    waypoint_id = row["ID"]

    # API-Request
    url = f"https://maps.googleapis.com/maps/api/elevation/json"
    params = {
        "locations": f"{latitude},{longitude}",
        "key": api_key
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        # Extract elevation
        elevation = data["results"][0]["elevation"]

        # Append the result
        results.append({
            "id": waypoint_id,
            "latitude": latitude,
            "longitude": longitude,
            "elevation": elevation
        })

    except (requests.exceptions.RequestException, KeyError, IndexError) as e:
        print(f"Fehler für ID {waypoint_id}: {e}")
        results.append({
            "id": waypoint_id,
            "latitude": latitude,
            "longitude": longitude,
            "elevation": None
        })

# transform the results in a data Frame
result_df = pd.DataFrame(results)

# Ausgabe des Ergebnisses
print(result_df)


                id   latitude  longitude    elevation
0       28022787.0  46.871293   6.581740  1390.655518
1       28022788.0  46.874111   6.589236  1291.659546
2       28022789.0  46.878365   6.598630  1194.116821
3       33636662.0  46.867863   6.573559  1421.337280
4       36789486.0  46.880193   6.612009  1263.661255
...            ...        ...        ...          ...
31760  497874972.0  47.713981   8.889910   419.983826
31761  497874993.0  47.711298   8.889283   440.823212
31762  497874994.0  47.713006   8.889728   424.057678
31763  497874995.0  47.710600   8.888011   455.662384
31764  497874996.0  47.709732   8.886718   494.598450

[31765 rows x 4 columns]


In [15]:
# Create table if it doesn't exist
table_name = "Google_Elevation"
query = f"""
    IF OBJECT_ID(N'dbo.{table_name}', N'U') IS NULL
    BEGIN
        CREATE TABLE {table_name} (
            id                      INT         NOT NULL,
            lat                     FLOAT       NOT NULL,
            lon                     FLOAT       NOT NULL,
            elevation               FLOAT       NOT NULL,
            PRIMARY KEY (id)
        );
    END
    """

conn = pymssql.connect(server, db_user, db_password, database)
cursor = conn.cursor()
cursor.execute(query)

conn.commit()
conn.close()

In [16]:
table_name = "Google_Elevation"

# Create connection string for SQLAlchemy
connection_string = f"mssql+pymssql://{db_user}:{db_password}@{server}/{database}"
engine = create_engine(connection_string)

# Ingest data to tabledatabase table
result_df.to_sql(table_name, con=engine, if_exists='replace', index=False)
print("DataFrame erfolgreich in die MSSQL-Datenbank geladen!")

DataFrame erfolgreich in die MSSQL-Datenbank geladen!
